In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# HomeoBot 
**A chatbot which gets symptoms from user and diagnoses symptomatically**

This makes usage of following:-

1. Long context window - Chatting with patients - get symptoms
2. Few-shot prompting - quick remedy suggesion
3. Voice command parser - ask quick remedies
4. Image understanding - Diagnosis by image understanding.
5. Audio understanding - get symptoms/quick remedies in audio format
6. Grounding - latest updates and medicines.
7. Generative AI agent - a chatbot 

**Install Python SDK**

In [2]:
# Remove conflicting packages from the Kaggle base environment.
!pip uninstall -qqy kfp jupyterlab libpysal thinc spacy fastai ydata-profiling google-cloud-bigquery google-generativeai
# Install langgraph and the packages used in this lab.
!pip install -qU 'langgraph==0.3.21' 'langchain-google-genai==2.1.2' 'langgraph-prebuilt==0.1.7'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.9/433.9 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 kB 11.2 MB/s eta 0:00:00


Install SDK

In [3]:
from google import genai
from google.genai import types

from IPython.display import HTML, Markdown, display

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:623: UserWarning: <built-in function any> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(


Set up retry helper

In [4]:
from google.api_core import retry


is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})

genai.models.Models.generate_content = retry.Retry(
    predicate=is_retriable)(genai.models.Models.generate_content)

Set up API key

In [5]:
import os
from kaggle_secrets import UserSecretsClient

GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

Get a model

In [6]:
from pprint import pprint

client = genai.Client(api_key=GOOGLE_API_KEY)

for model in client.models.list():
  if model.name == 'models/gemini-2.0-flash':
    pprint(model.to_json_dict())
    break

Automated retry

In [7]:
# Define a retry policy. The model might make multiple consecutive calls automatically
# for a complex query, this ensures the client retries if it hits quota limits.
from google.api_core import retry

is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})

if not hasattr(genai.models.Models.generate_content, '__wrapped__'):
  genai.models.Models.generate_content = retry.Retry(
      predicate=is_retriable)(genai.models.Models.generate_content)

The Python SDK uses a Client object to make requests to the API. The client lets you control which back-end to use (between the Gemini API and Vertex AI) and handles authentication (the API key).

The gemini-2.0-flash model has been selected here.

LangGraph applications are built around a graph structure. We will define an application graph that models the state transitions for your application. This will define a state schema, and an instance of that schema is propagated through the graph.

Each node in the graph represents an action or step that can be taken. Nodes will make changes to the state in some way through code that we have defined. These changes can be the result of invoking an LLM, by calling an API, or executing any logic that the node defines.

Each edge in the graph represents a transition between states, defining the flow of the program. Edge transitions can be fixed, for example if you define a text-only chatbot where output is always displayed to a user, you may always transition from chatbot -> user. The transitions can also be conditional, allowing you to add branching (like an if-else statement) or looping (like for or while loops).

# Define core instructions
State is a fundamental concept for a LangGraph app. A state object is passed between every node and transition in the app. Here we will define a state object, PatientState, that holds the conversation history, a structured diagnosis, and a flag indicating if the customer has finished with telling thier symptoms.

In Python, the LangGraph state object is a Python dictionary. You can provide a schema for this dictionary by defining it as a TypedDict.

Here we also define the system instruction that the Gemini model will use. You can capture tone and style here, as well as the playbook under which the chatbot should operate.

**Good Prompt**
A good Prompt always gives Correct and Accurate Answers. Let's promt it to make it a doctor.

In [8]:
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph.message import add_messages


class PatientState(TypedDict):
    """State representing the patient's symptomatic conversation."""

    # The chat conversation. This preserves the conversation history
    # between nodes. The `add_messages` annotation indicates to LangGraph
    # that state is updated by appending returned messages, not replacing
    # them.
    messages: Annotated[list, add_messages]

    # The customer's in-progress order.
    order: list[str]

    # Flag indicating that the diagnosis is done and completed.
    finished: bool


# The system instruction defines how the chatbot is expected to behave and includes
# rules for when to call different functions, as well as rules for the conversation, such
# as tone and what is permitted for discussion.
HOMEOBOT_SYSINT = (
    "system",  # 'system' indicates the message is a system instruction.
    "You are a HomeoBot, an interactive Homeopathic doctor. A human will talk to you about the "
    "symptoms he is facing and you will answer remedies about Systems menu(and only about "
    "symptomatic treatment - no off-topic discussion, but you can chat about the ill effects and their complictions). "
    "The customer will tell symptoms for 1 or more Systems from the menu, which you will structure "
    "and send to the ordering system after confirming the order with the human. "
    "\n\n"
    "Add items to the customer's order with add_to_order, and reset the order with clear_order. "
    "To see the contents of the symptoms so far, call get_order (this is shown to you, not the user) "
    "Always confirm_order with the user (double-check) before calling place_order. Calling confirm_order will "
    "display the symptoms to the user and returns their response to seeing the list. Their response may contain modifications. "
    "Always verify and respond with symptoms and thier fitness from the MENU before adding them to the diagnosis. "
    "If you are unsure a symptom matches those on the MENU, ask a question to clarify or redirect. "
    "You only have the Systems listed on the menu. "
    "Once the customer has finished telling symptoms , Call confirm_order to ensure it is correct then make "
    "any necessary updates and then call place_order. Once place_order has returned, thank the user and "
    "Print the prescription.!"
    "\n\n"
    "If any of the tools are unavailable, you can break the fourth wall and tell the user that "
    "they have not implemented them yet and should keep reading to do so.",
)

# This is the message with which the system opens the conversation.
WELCOME_MSG = "Welcome to the HomeoBot clinic. Type `q` to quit. May i Know your symptoms?"

# Defining a single turn chatboot
To illustrate how LangGraph works, the following program defines a chatbot node that will execute a single turn in a chat conversation using the instructions supplied.

Each node in the graph operates on the state object. The state (a Python dictionary) is passed as a parameter into the node (a function) and the new state is returned. This can be restated as pseudo-code, where state = node(state).

For the chatbot node, the state is updated by adding the new conversation message. The add_messages annotation on PatientState.messages indicates that messages are appended when returned from a node. Typically state is updated by replacement, but this annotation causes messages to behave differently.

In [9]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI

# Try using different models. The Gemini 2.0 flash model is highly
# capable, great with tools, and has a generous free tier. If you
# try the older 1.5 models, note that the `pro` models are better at
# complex multi-tool cases like this, but the `flash` models are
# faster and have more free quota.
# Check out the features and quota differences here:
#  - https://ai.google.dev/gemini-api/docs/models/gemini
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")


def chatbot(state: PatientState) -> PatientState:
    """The chatbot itself. A simple wrapper around the model's own chat interface."""
    message_history = [HOMEOBOT_SYSINT] + state["messages"]
    return {"messages": [llm.invoke(message_history)]}


# Set up the initial graph based on our state definition.
graph_builder = StateGraph(PatientState)

# Add the chatbot function to the app graph as a node called "chatbot".
graph_builder.add_node("chatbot", chatbot)

# Define the chatbot node as the app entrypoint.
graph_builder.add_edge(START, "chatbot")

chat_graph = graph_builder.compile()

Now that the graph is defined, we can run it. It only has one node, and one transition into that node, so it will transition from __start__ to chatbot, execute the chatbot node, and terminate.
To run the graph, we call invoke and pass an initial state object. In this case it begins with the user's initial message.

In [10]:
from pprint import pprint

user_msg = "Hello, what can you do?"
state = chat_graph.invoke({"messages": [user_msg]})

# The state object contains lots of information. Uncomment the pprint lines to see it all.
# pprint(state)

# Note that the final state now has 2 messages. Our HumanMessage, and an additional AIMessage.
for msg in state["messages"]:
    print(f"{type(msg).__name__}: {msg.content}")

HumanMessage: Hello, what can you do?
AIMessage: Hello! I am HomeoBot, an interactive Homeopathic doctor. I can help you find homeopathic remedies based on your symptoms. I will ask you about the symptoms you are experiencing in various systems, and then suggest remedies. Let's begin!


We could execute this in a Python loop, but for simplicity, manually invoke one more conversational turn. This second invocation takes the state from the first call and appends another user message to elicit another response from the chatbot.

In [11]:
user_msg = "Oh great, what type of medicines you prescribe?"

state["messages"].append(user_msg)
state = chat_graph.invoke(state)

# pprint(state)
for msg in state["messages"]:
    print(f"{type(msg).__name__}: {msg.content}")

HumanMessage: Hello, what can you do?
AIMessage: Hello! I am HomeoBot, an interactive Homeopathic doctor. I can help you find homeopathic remedies based on your symptoms. I will ask you about the symptoms you are experiencing in various systems, and then suggest remedies. Let's begin!
HumanMessage: Oh great, what type of medicines you prescribe?
AIMessage: I prescribe homeopathic remedies based on the symptoms you describe. These remedies are selected based on the principles of homeopathy, where "like cures like." To give you the best recommendations, I need you to describe your symptoms as accurately as possible. Which system are you experiencing issues with?

Here is the menu of systems I can assist you with:

**MENU**
*   **Digestive System**
*   **Respiratory System**
*   **Skin**
*   **Musculoskeletal System**
*   **Nervous System**
*   **Urogenital System**


# Adding a human node
Instead of repeatedly running the "graph" in a Python loop, we will use LangGraph to loop between nodes.

The human node will display the last message from the LLM to the user, and then prompt them for their next input. Here this is done using standard Python print and input functions along with rendering the chat to a display or audio, and accept input from a mic or on-screen keyboard.

The chatbot node function has also been updated to include the welcome message to start the conversation.

In [12]:
from langchain_core.messages.ai import AIMessage


def human_node(state: PatientState) -> PatientState:
    """Display the last model message to the user, and receive the user's input."""
    last_msg = state["messages"][-1]
    print("Model:", last_msg.content)

    user_input = input("User: ")

    # If it looks like the user is trying to quit, flag the conversation
    # as over.
    if user_input in {"q", "quit", "exit", "goodbye"}:
        state["finished"] = True

    return state | {"messages": [("user", user_input)]}


def chatbot_with_welcome_msg(state: PatientState) -> PatientState:
    """The chatbot itself. A wrapper around the model's own chat interface."""

    if state["messages"]:
        # If there are messages, continue the conversation with the Gemini model.
        new_output = llm.invoke([HOMEOBOT_SYSINT] + state["messages"])
    else:
        # If there are no messages, start with the welcome message.
        new_output = AIMessage(content=WELCOME_MSG)

    return state | {"messages": [new_output]}


# Start building a new graph.
graph_builder = StateGraph(PatientState)

# Add the chatbot and human nodes to the app graph.
graph_builder.add_node("chatbot", chatbot_with_welcome_msg)
graph_builder.add_node("human", human_node)

# Start with the chatbot again.
graph_builder.add_edge(START, "chatbot")

# The chatbot will always go to the human next.
graph_builder.add_edge("chatbot", "human");

Before you can run this, note that if you added an edge from human back to chatbot, the graph will cycle forever as there is no exit condition. One way to break the cycle is to add a check for a human input like q or quit and use that to break the loop.

In LangGraph, this is achieved with a conditional edge. This is similar to a regular graph transition, except a custom function is called to determine which edge to traverse.

Conditional edge functions take the state as input, and return a string representing the name of the node to which it will transition.

In [13]:
from typing import Literal


def maybe_exit_human_node(state: PatientState) -> Literal["chatbot", "__end__"]:
    """Route to the chatbot, unless it looks like the user is exiting."""
    if state.get("finished", False):
        return END
    else:
        return "chatbot"


graph_builder.add_conditional_edges("human", maybe_exit_human_node)

chat_with_human_graph = graph_builder.compile()

Run this new graph to see how the interaction loop is now captured within the graph. Input quit to exit the program.

You must uncomment the .invoke(...) line to run this step.

In [14]:
# The default recursion limit for traversing nodes is 25 - setting it higher means
# you can try a more complex order with multiple steps and round-trips (and you
# can chat for longer!)
config = {"recursion_limit": 100}

# Remember that this will loop forever, unless you input `q`, `quit` or one of the
# other exit terms defined in `human_node`.
# Uncomment this line to execute the graph:
#state = chat_with_human_graph.invoke({"messages": []}, config)

# Things to try:
#  - Just chat! There's no prescription yet.
#  - 'q' to exit.

# pprint(state)

# Add a "live" prescription menu
HomeoBot currently has no awareness of the symptoms of patient , so it will hallucinate a menu. One option would be to hard-code a menu into the system prompt. This would work well, but to simulate a system where the menu is more dynamic and could respond to fluctuating stock levels, you will put the menu into a custom tool.

There are two types of tools that this system will use. Stateless tools that can be run automatically, and stateful tools that modify the symptoms. The "get current menu" tool is stateless, in that it does not make any changes to the live order, so it can be called automatically.

In a LangGraph app, you can annotate Python functions as tools by applying the @tools annotation.

In [15]:
from langchain_core.tools import tool


@tool
def get_menu() -> str:
    """Provide the latest up-to-date menu."""
    # Note that this is just hard-coded text, but you could connect this to a live stock
    # database, or you could use Gemini's multi-modal capabilities and take live photos of
    # your cafe's chalk menu or the products on the counter and assmble them into an input.

    return """
    MENU:
    Digestive System:
    Acid Reflux
    Bloating
    Constipation
    Diarrhea
    Nausea

    Respiratory System:
    Cough (Dry)
    Cough (Productive)
    Shortness of Breath
    Sinus Congestion
    Sore Throat

    Skin:
    Eczema
    Hives
    Itching
    Rashes
    Urticaria

    Sleep:
    Insomnia
    Restlessness

    Musculoskeletal System:
    Back Pain
    Joint Pain
    Muscle Cramps
    Stiffness
    """

Now add the new tool to the graph. The get_menu tool is wrapped in a ToolNode that handles calling the tool and passing the response as a message through the graph. The tools are also bound to the llm object so that the underlying model knows they exist. As you now have a different llm object to invoke, you need to update the chatbot node so that it is aware of the tools.

In [16]:
from langgraph.prebuilt import ToolNode


# Define the tools and create a "tools" node.
tools = [get_menu]
tool_node = ToolNode(tools)

# Attach the tools to the model so that it knows what it can call.
llm_with_tools = llm.bind_tools(tools)


def maybe_route_to_tools(state: PatientState) -> Literal["tools", "human"]:
    """Route between human or tool nodes, depending if a tool call is made."""
    if not (msgs := state.get("messages", [])):
        raise ValueError(f"No messages found when parsing state: {state}")

    # Only route based on the last message.
    msg = msgs[-1]

    # When the chatbot returns tool_calls, route to the "tools" node.
    if hasattr(msg, "tool_calls") and len(msg.tool_calls) > 0:
        return "tools"
    else:
        return "human"


def chatbot_with_tools(state: PatientState) -> PatientState:
    """The chatbot with tools. A simple wrapper around the model's own chat interface."""
    defaults = {"order": [], "finished": False}

    if state["messages"]:
        new_output = llm_with_tools.invoke([HOMEOBOT_SYSINT] + state["messages"])
    else:
        new_output = AIMessage(content=WELCOME_MSG)

    # Set up some defaults if not already set, then pass through the provided state,
    # overriding only the "messages" field.
    return defaults | state | {"messages": [new_output]}


graph_builder = StateGraph(PatientState)

# Add the nodes, including the new tool_node.
graph_builder.add_node("chatbot", chatbot_with_tools)
graph_builder.add_node("human", human_node)
graph_builder.add_node("tools", tool_node)

# Chatbot may go to tools, or human.
graph_builder.add_conditional_edges("chatbot", maybe_route_to_tools)
# Human may go back to chatbot, or exit.
graph_builder.add_conditional_edges("human", maybe_exit_human_node)

# Tools always route back to chat afterwards.
graph_builder.add_edge("tools", "chatbot")

graph_builder.add_edge(START, "chatbot")
graph_with_menu = graph_builder.compile()

Now run the new graph to see how the model uses the menu.

You must uncomment the .invoke(...) line to run this step.

In [17]:
# Remember that you have not implemented ordering yet, so this will loop forever,
# unless you input `q`, `quit` or one of the other exit terms defined in the
# `human_node`.
# Uncomment this line to execute the graph:
# state = graph_with_menu.invoke({"messages": []}, config)

# Things to try:
# - How to treat nausea
# - Medecines to treat cough
# - 'q' to exit.


# pprint(state)

# Handle Prescription
To build up an order during the chat conversation, you will need to update the state to track the diagnosis, and provide simple tools that update this state. These need to be explicit as the model should not directly have access to the apps internal state, or it risks being manipulated arbitrarily.

The ordering tools will be added as stubs in a separate node so that you can edit the state directly. Using the @tool annotation is still a handy way to define their schema, so the ordering tools below are implemented as empty Python functions.

In [18]:
from collections.abc import Iterable
from random import randint

from langchain_core.messages.tool import ToolMessage

# These functions have no body; LangGraph does not allow @tools to update
# the conversation state, so you will implement a separate node to handle
# state updates. Using @tools is still very convenient for defining the tool
# schema, so empty functions have been defined that will be bound to the LLM
# but their implementation is deferred to the order_node.


@tool
def add_to_order(symptom: str, modifiers: Iterable[str]) -> str:
    """Adds the specified symptom to the patient's symptom, including any modifiers.

    Returns:
      The updated order in progress.
    """


@tool
def confirm_order() -> str:
    """Asks the customer if the symptoms are correct.

    Returns:
      The user's free-text response.
    """


@tool
def get_order() -> str:
    """Returns the users symptoms so far. One item per line."""


@tool
def clear_order():
    """Removes all items from the user's symptoms."""


@tool
def place_order() -> int:
    """Sends the symptoms to the doctor for fulfillment.

    Returns:
      The estimated number of minutes until the prescription is ready.
    """


def order_node(state: PatientState) -> PatientState:
    """The ordering node. This is where the patient state is manipulated."""
    tool_msg = state.get("messages", [])[-1]
    order = state.get("order", [])
    outbound_msgs = []
    order_placed = False

    for tool_call in tool_msg.tool_calls:

        if tool_call["name"] == "add_to_order":

            # Each order item is just a string. This is where it assembled as "symptom (modifiers, ...)".
            modifiers = tool_call["args"]["modifiers"]
            modifier_str = ", ".join(modifiers) if modifiers else "no modifiers"

            order.append(f'{tool_call["args"]["drink"]} ({modifier_str})')
            response = "\n".join(order)

        elif tool_call["name"] == "confirm_order":

            # We could entrust the LLM to do order confirmation, but it is a good practice to
            # show the user the exact data that comprises their order so that what they confirm
            # precisely matches the order that goes to the kitchen - avoiding hallucination
            # or reality skew.

            
            print("Your order:")
            if not order:
                print("  (no items)")

            for symptom in order:
                print(f"  {symptom}")

            response = input("Is this correct? ")

        elif tool_call["name"] == "get_order":

            response = "\n".join(order) if order else "(no order)"

        elif tool_call["name"] == "clear_order":

            order.clear()
            response = None

        elif tool_call["name"] == "place_order":

            order_text = "\n".join(order)
            print("Sending symptoms to doctor!")
            print(order_text)

            # TODO(you!): Prescribe.
            order_placed = True
            response = randint(1, 5)  # ETA in minutes

        else:
            raise NotImplementedError(f'Unknown tool call: {tool_call["name"]}')

        # Record the tool results as tool messages.
        outbound_msgs.append(
            ToolMessage(
                content=response,
                name=tool_call["name"],
                tool_call_id=tool_call["id"],
            )
        )

    return {"messages": outbound_msgs, "order": order, "finished": order_placed}


def maybe_route_to_tools(state: PatientState) -> str:
    """Route between chat and tool nodes if a tool call is made."""
    if not (msgs := state.get("messages", [])):
        raise ValueError(f"No messages found when parsing state: {state}")

    msg = msgs[-1]

    if state.get("finished", False):
        # When an order is placed, exit the app. The system instruction indicates
        # that the chatbot should say thanks and goodbye at this point, so we can exit
        # cleanly.
        return END

    elif hasattr(msg, "tool_calls") and len(msg.tool_calls) > 0:
        # Route to `tools` node for any automated tool calls first.
        if any(
            tool["name"] in tool_node.tools_by_name.keys() for tool in msg.tool_calls
        ):
            return "tools"
        else:
            return "ordering"

    else:
        return "human"

Now define the graph. The LLM needs to know about the tools too, so that it can invoke them. Here you set up 2 sets of tools corresponding to the nodes under which they operate: automated and ordering.

In [19]:
# Auto-tools will be invoked automatically by the ToolNode
auto_tools = [get_menu]
tool_node = ToolNode(auto_tools)

# Order-tools will be handled by the order node.
order_tools = [add_to_order, confirm_order, get_order, clear_order, place_order]

# The LLM needs to know about all of the tools, so specify everything here.
llm_with_tools = llm.bind_tools(auto_tools + order_tools)


graph_builder = StateGraph(PatientState)

# Nodes
graph_builder.add_node("chatbot", chatbot_with_tools)
graph_builder.add_node("human", human_node)
graph_builder.add_node("tools", tool_node)
graph_builder.add_node("ordering", order_node)

# Chatbot -> {ordering, tools, human, END}
graph_builder.add_conditional_edges("chatbot", maybe_route_to_tools)
# Human -> {chatbot, END}
graph_builder.add_conditional_edges("human", maybe_exit_human_node)

# Tools (both kinds) always route back to chat afterwards.
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge("ordering", "chatbot")

graph_builder.add_edge(START, "chatbot")
graph_with_order_tools = graph_builder.compile()


Now run the complete ordering system graph.

You must uncomment the .invoke(...) line to run this step.

In [20]:
# Uncomment this line to execute the graph:
# state = graph_with_order_tools.invoke({"messages": []}, config)

# Things to try:
# - Symptom for cough , dizziness

# - Note that the graph should naturally exit after placing an order.

# pprint(state)

The patient state has been captured both in the place_order function and in the final conversational state returned from executing the graph. This iillustrates how you can integrate your own systems to a graph app, as well as collect the final results of executing such an app.

In [21]:
# Uncomment this once you have run the graph from the previous cell.
# pprint(state["order"])

# LIMITATION

**Homeopathic medicines are given on basis of indivisuality. So for every disease and for every person , medicine to be given is different , selection of accurate medicine becomes tough. On basis of changing history of symptoms of patients one needs to change medicines frequently , power needs to be changed frequently which willbe tough for AI to decide when to change power.We need to give intercurrent remedies to boost curing action of curing disease which will be hard for AI to cop up with this. Selection of rare remedies is difficult.**


# BENEFIT
**Homeopathic medicines have no side effects.**

# Voice command parser to recognize symptoms through speech

This notebook demonstrates a chatbot that takes short voice commands via audio upload,transcribes them using Google Cloud Speech-to-Text, and then parses them into a structured JSON format (intent and parameters).

Here we make chatbot that takes short voice commands via audio upload,transcribes them using Google Cloud Speech-to-Text, and then parses them into a structured JSON format (intent and parameters).

Installing dependencies

In [22]:
!pip install google-cloud-speech google-generativeai ipywidgets ipython --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.1/334.1 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 44.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-google-genai 2.1.2 requires google-ai-generativelanguage<0.7.0,>=0.6.16, but you have google-ai-generativelanguage 0.6.15 which is incompatible.


Import libraries

In [23]:
import ipywidgets as widgets
from IPython.display import display, Audio, clear_output
from kaggle_secrets import UserSecretsClient
import google.cloud.speech as speech

import os
import json
import io
import time # For potential retries or delays

print("Libraries installed and imported.")

Libraries installed and imported.


Configre speech-to-text

In [24]:
speech_client = None
speech_client_initialized = False
try:
    # OPTIONAL: If default auth fails, uncomment and set path to your Service Account Key JSON

    speech_client = speech.SpeechClient()
    # Try a simple API call to check auth (listing recognizers requires specific permissions, maybe too much)
    # A better check is just proceeding and catching errors during the 'recognize' call.
    print("Google Cloud Speech client initialized (attempted).")
    speech_client_initialized = True

except Exception as e:
    print(f"Error initializing Google Cloud Speech client: {e}")
    print("Speech-to-Text might require setting the GOOGLE_APPLICATION_CREDENTIALS environment variable.")
    print("See comments in the code above regarding Service Account Keys.")

Google Cloud Speech client initialized (attempted).


Widget setup

In [25]:
file_upload = widgets.FileUpload(
    accept='.wav,.flac,.mp3,.ogg,.m4a', # Check Speech-to-Text docs for best formats
    multiple=False,
    description='Upload Audio Command'
)

process_button = widgets.Button(
    description="Parse Command",
    button_style='info',
    tooltip='Upload an audio file first, then click to process',
    icon='microphone'
    )
process_button.disabled = True

output_area = widgets.Output()

def on_file_upload_change(change):
    if change['new']:
        process_button.disabled = False
        process_button.button_style = 'success'
    else:
        process_button.disabled = True
        process_button.button_style = 'info'

file_upload.observe(on_file_upload_change, names='value')



Logic with **Few-shot prompts**.
This will help recognize voice easily.

In [26]:
def parse_command_from_audio_google(b): # Button click handler
    with output_area:
        clear_output(wait=True)
        print("Processing...")

        if not genai_client_initialized:
             print("Error: Google GenAI client not initialized. Check API Key setup.")
             return
        if not speech_client_initialized:
             print("Error: Google Speech client not initialized. Check Authentication setup / Service Account.")
             # You might allow proceeding if only transcription fails, but parsing needs input
             # return

        if not file_upload.value:
            print("Error: Please upload an audio file first.")
            return

        # --- Get uploaded file data ---
        uploaded_file_info = list(file_upload.value.values())[0]
        file_content = uploaded_file_info['content'] # Bytes
        original_file_name = uploaded_file_info['name']
        print(f"Recieved file: {original_file_name}")

        # Display audio player
        display(Audio(data=file_content))

        # --- Capability 1: Audio Understanding (Speech-to-Text) ---
        transcribed_text = None
        if speech_client: # Proceed only if client seems initialized
            try:
                print("\nTranscribing audio using Google Cloud Speech-to-Text...")

                # Prepare audio object for the API
                audio = speech.RecognitionAudio(content=file_content)

                # Configure recognition settings
                
                # Let's try a simpler config first, add more details if needed.
                config = speech.RecognitionConfig(
                    # encoding=speech.RecognitionConfig.AudioEncoding.LINEAR16, # Specify if known (e.g., for WAV)
                    # sample_rate_hertz=16000, # Specify if known
                    language_code="en-US",  # BCP-47 language tag
                    enable_automatic_punctuation=True # Helpful for parsing
                )

                # Perform synchronous recognition (best for short audio)
                response = speech_client.recognize(config=config, audio=audio)

                # --- Extract transcript ---
                if response.results:
                    # Concatenate transcripts from all results/alternatives if needed
                    transcribed_text = " ".join(
                        [result.alternatives[0].transcript for result in response.results]
                    ).strip()
                    if transcribed_text:
                         print(f"Transcription successful:\n---> '{transcribed_text}'")
                    else:
                         print("Transcription returned empty result.")
                else:
                    print("Transcription returned no results.")

            except Exception as e:
                print(f"Error during Google Cloud Speech-to-Text transcription: {e}")
                print("   Check API enablement, permissions, audio format, or Service Account setup.")
                # Optionally stop here, or try parsing if there was old text?
                # return
        else:
             print("Skipping transcription because Speech client failed to initialize.")


        # ---  Few-shot Prompting & Structured Output (JSON - via Prompting) ---
        if transcribed_text and genai_client_initialized:
            print("\nParsing command using Gemini with Few-Shot Prompting...")
            try:
                # Construct the prompt history for Gemini
                # Structure: List of {'role': 'user'/'model', 'parts': [text]}
                prompt_history = [
                    # System Instruction (as the first 'user' turn often works well)
                     {'role': 'user',
                      'parts': ["""You are a command parser assistant. Analyze the user's transcribed text
                                     and extract the command intent and any relevant parameters.
                                     Respond ONLY with a single, valid JSON object containing 'intent' and 'parameters'.
                                     The 'parameters' should be an object containing key-value pairs.
                                     Do NOT include any text before or after the JSON object, including markdown markers like ```json.
                                     If no clear command is found, respond with {"intent": "unknown", "parameters": {}}.
                                     Follow the examples precisely."""]
                     },
                     {'role': 'model', # Acknowledge the instruction
                      'parts': ["Okay, I understand. I will analyze the text and respond only with the JSON object."]
                     },
                    # Example 1
                    {'role': 'user',
                     'parts': ["Medicine for headache"]
                    },
                    {'role': 'model',
                     'parts': [json.dumps({"intent": "get_medicine_name", "parameters": {"disease": "headache"}})]
                    },
                     # Example 2
                    {'role': 'user',
                     'parts': ["When to prescribe Arnica 1M"]
                    },
                    {'role': 'model',
                     'parts': [json.dumps({"intent": "get_use", "parameters": {"medicine": "Arnica" , "power": "1M"}})]
                    },
                     # Example 3
                    {'role': 'user',
                     'parts': ["Pain in renal pelvis"]
                    },
                    {'role': 'model',
                     'parts': [json.dumps({"intent": "get_diagnosis", "parameters": {"part": "renal pelvis", "symptom": "pain"}})]
                    },
                     # Example 4 (No clear command)
                    {'role': 'user',
                     'parts': ["ummm I think maybe later"]
                    },
                    {'role': 'model',
                     'parts': [json.dumps({"intent": "unknown", "parameters": {}})]
                    },
                    # Actual user input (last user turn)
                     {'role': 'user',
                      'parts': [transcribed_text]
                     }
                ]

                # Generate content using the Gemini model
                # print(f"Sending to Gemini: {prompt_history}") # Debugging: See the exact prompt
                generation_config = genai.types.GenerationConfig(
                    temperature=0.1 # Low temp for predictable parsing
                    # max_output_tokens=... # Optional
                )
                response = gemini_model.generate_content(
                    prompt_history,
                    generation_config=generation_config
                    )

                # --- Extract and Parse JSON response ---
                # We rely heavily on the prompt instructing it to ONLY output JSON.
                # Add cleanup if necessary (e.g., stripping markdown)
                parsed_command_str = response.text.strip()

                # Basic cleanup (in case model still adds markdown)
                if parsed_command_str.startswith("```json"):
                    parsed_command_str = parsed_command_str[7:]
                if parsed_command_str.endswith("```"):
                    parsed_command_str = parsed_command_str[:-3]
                parsed_command_str = parsed_command_str.strip()

                # Attempt to parse the cleaned string as JSON
                parsed_command_json = json.loads(parsed_command_str)

                print("Command parsing successful:")
                # Pretty print the JSON
                print(json.dumps(parsed_command_json, indent=2))

            except json.JSONDecodeError as e:
                 print(f"Error: Failed to decode JSON response from Gemini: {e}")
                 print(f"Gemini raw response:\n---\n{response.text}\n---")
            except Exception as e:
                # Catch potential errors from generate_content (e.g., safety blocks)
                print(f"Error during command parsing with Gemini: {e}")
                # Print response parts if available for debugging safety issues etc.
                try:
                    print(f"Gemini full response object: {response}")
                    # print(f"Prompt Feedback: {response.prompt_feedback}")
                    # print(f"Candidates: {response.candidates}") # Check finish_reason
                except Exception:
                    pass # Ignore errors trying to print debug info

        elif not transcribed_text:
             print("\nSkipping command parsing because transcription failed or was empty.")
        elif not genai_client_initialized:
             print("\nSkipping command parsing because Gemini client is not initialized.")


        # --- Cleanup and Reset ---
        print("\nProcessing complete. Ready for next command.")
        # Reset file upload widget
        file_upload.value.clear()
        file_upload._counter = 0


# Link button click to the processing function
process_button.on_click(parse_command_from_audio_google)


Display user interface

In [27]:
print("Voice Command Parser Interface")
print("Upload an audio file (.wav, .flac recommended) containing a command, then click 'Parse Command'.")
#Uncomment this to upload audio
#display(file_upload)
#display(process_button)
#display(output_area)

Voice Command Parser Interface
Upload an audio file (.wav, .flac recommended) containing a command, then click 'Parse Command'.


# Image and Text Command Parser to get detail symptomatic diagnosis

The notebook displays the parsed JSON command/query, potentially incorporating information derived from the image.

Install Libraries

In [28]:
!pip install google-generativeai ipywidgets Pillow --quiet

Import libraries

In [29]:
import ipywidgets as widgets
from IPython.display import display, clear_output, Image as IPImage # Display image
from kaggle_secrets import UserSecretsClient
from PIL import Image as PILImage # For image handling
import io
import os
import json

print("Libraries installed and imported.")

Libraries installed and imported.


Set up GUI

In [30]:
image_upload = widgets.FileUpload(
    accept='image/*', # Accept any image type
    multiple=False,
    description='Upload Image'
)

text_input = widgets.Textarea(
    value='',
    placeholder='Type command related to image (e.g., "Describe this", "What color is the car?") or unrelated (e.g., "Set timer 10 min")',
    description='Command:',
    layout={'height': '80px', 'width': '80%'}
)

process_button = widgets.Button(
    description="Parse Image Command",
    button_style='info',
    tooltip='Upload image and type command, then click',
    icon='cogs' # Changed icon
    )
process_button.disabled = True # Disabled until both inputs are ready

output_area = widgets.Output()

# Function to enable button only when both image and text are provided
def check_inputs(*args):
    if image_upload.value and text_input.value.strip():
        process_button.disabled = False
        process_button.button_style = 'success'
    else:
        process_button.disabled = True
        process_button.button_style = 'info'

image_upload.observe(check_inputs, names='value')
text_input.observe(check_inputs, names='value')


Logic with **Few-shot** prompts

In [31]:
def parse_image_command_google(b): # Button click handler
    with output_area:
        clear_output(wait=True)
        print("Processing...")

        if not genai_client_initialized:
             print("Error: Google GenAI client not initialized. Check API Key setup.")
             return

        if not image_upload.value or not text_input.value.strip():
            print("Error: Please upload an image AND type a command.")
            return

        # --- Get uploaded image data and text ---
        uploaded_image_info = list(image_upload.value.values())[0]
        image_content = uploaded_image_info['content'] # Bytes
        image_name = uploaded_image_info['name']
        user_text_command = text_input.value.strip()

        print(f"Received Image: {image_name}")
        print(f"Received Text Command: '{user_text_command}'")

        # Display uploaded image for verification
        display(IPImage(data=image_content, width=200)) # Display using IPython.display

        # --- Prepare Image for Gemini ---
        try:
            # Use PIL to open the image from bytes - also serves as basic validation
            img_pil = PILImage.open(io.BytesIO(image_content))
            # Gemini library expects PIL image objects directly for multi-modal input
        except Exception as e:
            print(f"Error opening image file: {e}. Please upload a valid image format (JPG, PNG etc.).")
            # Cleanup widgets
            image_upload.value.clear()
            image_upload._counter = 0
            check_inputs() # Update button state
            return

        # ---  Few-shot Prompting (Multi-modal) & Structured Output ---
        print("\nParsing command using Gemini (Vision) with Few-Shot Prompting...")
        try:
            # Construct the prompt history for Gemini (Multi-modal)
            # Few-shot examples now need to *represent* image context conceptually
            prompt_parts = [
                 # System Instruction / Task Definition
                 """You are an advanced command parser assistant capable of understanding images.
Analyze the user's text command in the context of the provided image.
Extract the command intent and relevant parameters, potentially deriving information from the image.
Respond ONLY with a single, valid JSON object containing 'intent' and 'parameters'.
The 'parameters' should be an object containing key-value pairs.
Do NOT include any text before or after the JSON object, including markdown markers like ```json.
If the command is unrelated to the image, parse it based on text only.
If no clear command is found, respond with {"intent": "unknown", "parameters": {}}.
Follow the examples precisely.
""",
                 # Example 1 (Image relevant - Object identification)
                 "<Example Image: Dark patches on skin>", # Simulate image context for prompt
                 "What disease this is?",
                 json.dumps({"intent": "get_diagnosis", "parameters": {"Organ": "skin", "attribute": "dark spots"}}),

                 # Example 2 (Image relevant - Action on object)
                 "<Example Image: Person with bulging eyes>",
                 "This symptom is shown by which type of patients",
                 json.dumps({"intent": "Get-diagnosis", "parameters": {"organ": "thyroid", "identifier": "bulging_eyes"}}),

                 # Example 3 (Image relevant - Information extraction)
                 "<Example Image: A swollen hand >",
                 "Remedies to reduce swelling",
                 json.dumps({"intent": "get_diagnosis", "parameters": {"organ": "hand", "symptom": "swelling"}}),

                 # Example 4 (Image relevant - Description)
                 "<Example Image: A person having nausea>",
                 "Treatment for nausea",
                 json.dumps({"intent": "describe_image", "parameters": {"subject": "Measures to prevent nausea"}}),

                  # Example 5 (Ambiguous)
                 "<Example Image: A blurry photo>",
                 "What was that?",
                 json.dumps({"intent": "unknown", "parameters": {}}),

                 # --- Actual User Input (Image + Text) ---
                 # The Gemini library requires the actual image object here
                 img_pil, # The actual PIL Image object
                 user_text_command # The actual text command from the user
            ]

            # Generate content using the Gemini vision model
            generation_config = genai.types.GenerationConfig(
                temperature=0.1 # Low temp for predictable parsing
            )
            # Send the combined parts list
            response = gemini_model.generate_content(
                prompt_parts,
                generation_config=generation_config
                )

            # --- Extract and Parse JSON response ---
            parsed_command_str = response.text.strip()

            # Basic cleanup (in case model still adds markdown)
            if parsed_command_str.startswith("```json"):
                parsed_command_str = parsed_command_str[7:]
            if parsed_command_str.endswith("```"):
                parsed_command_str = parsed_command_str[:-3]
            parsed_command_str = parsed_command_str.strip()

            if not parsed_command_str:
                 print("Error: Gemini returned an empty response.")
                 raise ValueError("Empty response received")

            # Attempt to parse the cleaned string as JSON
            parsed_command_json = json.loads(parsed_command_str)

            print("Command parsing successful:")
            # Pretty print the JSON
            print(json.dumps(parsed_command_json, indent=2))

        except json.JSONDecodeError as e:
             print(f"Error: Failed to decode JSON response from Gemini: {e}")
             print(f"Gemini raw response:\n---\n{response.text}\n---")
        except Exception as e:
            print(f"Error during command parsing with Gemini: {e}")
            try:
                # Try printing safety feedback if available (often helpful)
                print(f"Gemini prompt feedback: {response.prompt_feedback}")
                # print(f"Candidates: {response.candidates}") # Check finish_reason
            except Exception:
                pass # Ignore errors trying to print debug info

        # --- Cleanup and Reset ---
        print("\nProcessing complete. Ready for next command.")
        # Reset widgets
        image_upload.value.clear()
        image_upload._counter = 0
        text_input.value = '' # Clear text area
        # Button state updated by observer


# Link button click to the processing function
process_button.on_click(parse_image_command_google)


Display UI

In [32]:
print("Image + Text Command Parser Interface")

print("Upload an image AND type a command, then click 'Parse Image Command'.")
#Uncomment this line to upload image
#display(widgets.VBox([image_upload, text_input, process_button, output_area]))

Image + Text Command Parser Interface
Upload an image AND type a command, then click 'Parse Image Command'.


# Adding Ground search for additional up to date information.

Install SDK

In [33]:
# Uninstall packages from Kaggle base image that are not needed.
!pip uninstall -qy jupyterlab jupyterlab-lsp
# Install the google-genai SDK for this codelab.
!pip install -qU 'google-genai==1.7.0'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.7/144.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.9/100.9 kB 5.6 MB/s eta 0:00:00


# Make a request
To enable search grounding, you specify it as a tool: google_search. Like other tools, this is supplied as a parameter in GenerateContentConfig, and can be passed to generate_content calls as well as chats.create (for all chat turns) or chat.send_message (for specific turns).

In [34]:
#Now run the query with search grounding enabled.
config_with_search = types.GenerateContentConfig(
    tools=[types.Tool(google_search=types.GoogleSearch())],
)

def query_with_grounding():
    response = client.models.generate_content(
        model='gemini-2.0-flash',
        contents="Homeopathic medicines latest updates",
        config=config_with_search,
    )
    return response.candidates[0]


rc = query_with_grounding()
Markdown(rc.content.parts[0].text)

Here's a summary of recent updates and perspectives on homeopathic medicine:

**1. Regulatory Updates & Safety Concerns:**

*   **FDA Scrutiny:** The FDA is increasing its regulatory actions on homeopathic drug products, particularly those posing the greatest risk to patients. They are focusing on products with reported injuries, significant quality issues, or those intended for serious conditions or vulnerable populations.
*   **No FDA Approvals:** It's important to note that currently, no homeopathic products have been approved by the FDA for any use. This means they haven't been evaluated for safety and effectiveness.
*   **FTC Enforcement:** The Federal Trade Commission (FTC) requires that advertising claims for homeopathic drugs be backed by scientific evidence.
*   **Warning Letters:** Since 2020, the FDA has issued numerous warning letters to companies for violations, including quality issues like sterility concerns.
*   **Risk-Based Enforcement:** The FDA is using a risk-based approach to prioritize regulatory actions, focusing on products that pose the most significant risk to patients.
*   **Regulation as Drugs:** In the U.S., homeopathic products are regulated as drugs under the Federal Food, Drug, and Cosmetic Act.
*   **Contaminants:** Some homeopathic products may contain substantial amounts of active ingredients, including heavy metals or toxins, which can cause adverse effects.

**2. Effectiveness & Scientific Evidence:**

*   **Lack of Evidence:** Major reviews and assessments, including those by the Australian government's National Health and Medical Research Council and the EU in 2017, have concluded that there is no reliable scientific evidence that homeopathy is effective for any health condition.
*   **Placebo Effect:** A report by the UK's House of Commons Science and Technology Committee stated that homeopathic remedies perform no better than placebos.
*   **Implausibility:** Mainstream science does not accept the ideas behind homeopathy, such as "like-cures-like" and the increasing potency of highly diluted substances.
*   **No Prevention of Disease:** There's no evidence that homeopathy can prevent diseases like malaria.

**3. Research & Publications:**

*   **Homeopathy Research Institute (HRI):** This UK-based charity promotes research in homeopathy.
*   **Advancements in Homeopathic Research:** This journal publishes research work and clinical studies in homeopathy. Recent publications include research on back pain, standardization of homeopathic products, and the efficacy of homeopathic medicine for vitiligo.
*   **Positive Replicated Experiments:** A review of basic research on highly dilute homeopathic medicines found a significant number of replicated experiments with positive results.
*   **Nanoparticles:** There's growing evidence that nanoparticles play a role in how homeopathic medicines work.

**4. Use & Integration:**

*   **Reduced Use of Conventional Drugs:** Some research suggests that integrating homeopathy into medical practice may reduce the use of drugs like antibiotics.
*   **Geographic Spread:** Homeopathy is geographically widespread and increasing in popularity, and has shown itself to be safe and effective for a range of conditions.
*   **Royal Patronage:** King Charles III is a known supporter of homeopathy.
*   **Homeopathy and Weight Loss:** Homeopathy is being explored as a gentle alternative to harsh medication for weight loss.

**5. Global Perspectives:**

*   **France:** France has abolished state funding for homeopathy.
*   **WHO Traditional Medicine Global Summit:** Members of the European Committee for Homeopathy (ECH) attended the WHO Traditional Medicine Global Summit 2023.

**6. Other News:**

*   **New Homeopathy College:** A new private homeopathy college has been approved in Chittoor, India.
*   **World Homeopathy Day:** World Homeopathy Day was observed on April 10, 2025.
*   **Regulations in India:** The National Commission for Homoeopathy in India continues to update regulations regarding graduate and postgraduate courses.

It's important to note that while some studies suggest potential benefits, the prevailing scientific consensus is that homeopathy lacks sufficient evidence of effectiveness and may pose risks if used in place of conventional medical treatment.


# Response metadata
When search grounding is used, the model returns extra metadata that includes links to search suggestions, supporting documents and information on how the supporting documents were used.

Each "grounding chunk" represents information retrieved from Google Search that was used in the grounded generation request. Following the URI will take you to the source.

In [35]:
while not rc.grounding_metadata.grounding_supports or not rc.grounding_metadata.grounding_chunks:
    # If incomplete grounding data was returned, retry.
    rc = query_with_grounding()

chunks = rc.grounding_metadata.grounding_chunks
for chunk in chunks:
    print(f'{chunk.web.title}: {chunk.web.uri}')

avalere.com: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAJDHK4-h1LHlW2iSwKSqtQyaZZdBpYh8Doya1q1ATEsIRDzCjpyBQ-NRjd5R8nSxqXqp2qxl4RNQ0aGAc0nfCdnUg-8IMw1My--q4ZOTDg6bxYeQ1kzNT8iNHgYgjK_fp9USVPYty-MsPPjqHwocvo_QiZopzEYHwQiKjutMdiO9cOJAZWNwbgrG_mBPUYT2fgrEA==
fda.gov: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAJUU98nzMk0Jex0iCkAxo5bY24mzF2vIbCBMuLiznjKDMtde4JC9p1EnS-OmjC4n0kFoFTdNqY43LPqCxIQpJqbC28cF53b-4GfODLVGuAZqyWtXGg2VPZECS0uzrm2jXcstdcAvrqk3JyWj_bVfHGaoYQ_x4zo_NwMks1MaNMq3g==
mintz.com: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAI2dt5-wxP51byAmNsRDlOJwSr-FC8E156wEK_Ge7eC2dlwnFWXu21tI6ktBk7dCuDHiN0b_zPa43V8jqgDeiRVo5tMGYYMnfo0bbAXpL9TStqlxkwk798J5UmodgcwLkKEu3Y8bf_kxLSGRyKUVTvFydEh0Xc6JiRSqL3MaL56AsPx-vXfjAyb2fLfPbTfGrmBLkBCmqhF1zmpt-jNI3zrzWkjTsx_H2EaC37fH1SWd2I=
nih.gov: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAL9XwzNDInFnSRaiWr8h_fMoS8fZYGPn5lJ_3xieYAAAJJtXNaYXXez9PxbiVBLr_vQXKT

As part of the response, there is a standalone styled HTML content block that you use to link back to relevant search suggestions related to the generation.

In [36]:
HTML(rc.grounding_metadata.search_entry_point.rendered_content)

The grounding_supports in the metadata provide a way for you to correlate the grounding chunks used to the generated output text.

In [37]:
from pprint import pprint

supports = rc.grounding_metadata.grounding_supports
for support in supports:
    pprint(support.to_json_dict())

{'confidence_scores': [0.6506869],
 'grounding_chunk_indices': [0],
 'segment': {'end_index': 427,
             'start_index': 278,
             'text': 'They are focusing on products with reported injuries, '
                     'significant quality issues, or those intended for '
                     'serious conditions or vulnerable populations.'}}
{'confidence_scores': [0.9235832],
 'grounding_chunk_indices': [1],
 'segment': {'end_index': 559,
             'start_index': 428,
             'text': "*   **No FDA Approvals:** It's important to note that "
                     'currently, no homeopathic products have been approved by '
                     'the FDA for any use.'}}
{'confidence_scores': [0.7782934],
 'grounding_chunk_indices': [1],
 'segment': {'end_index': 628,
             'start_index': 560,
             'text': "This means they haven't been evaluated for safety and "
                     'effectiveness.'}}
{'confidence_scores': [0.6760745, 0.9087099],
 'grounding_

These supports can be used to highlight text in the response, or build tables of footnotes.

In [38]:
import io

markdown_buffer = io.StringIO()

# Print the text with footnote markers.
markdown_buffer.write("Supported text:\n\n")
for support in supports:
    markdown_buffer.write(" * ")
    markdown_buffer.write(
        rc.content.parts[0].text[support.segment.start_index : support.segment.end_index]
    )

    for i in support.grounding_chunk_indices:
        chunk = chunks[i].web
        markdown_buffer.write(f"<sup>[{i+1}]</sup>")

    markdown_buffer.write("\n\n")


# And print the footnotes.
markdown_buffer.write("Citations:\n\n")
for i, chunk in enumerate(chunks, start=1):
    markdown_buffer.write(f"{i}. [{chunk.web.title}]({chunk.web.uri})\n")


Markdown(markdown_buffer.getvalue())

Supported text:

 * They are focusing on products with reported injuries, significant quality issues, or those intended for serious conditions or vulnerable populations.<sup>[1]</sup>

 * *   **No FDA Approvals:** It's important to note that currently, no homeopathic products have been approved by the FDA for any use.<sup>[2]</sup>

 * This means they haven't been evaluated for safety and effectiveness.<sup>[2]</sup>

 * *   **FTC Enforcement:** The Federal Trade Commission (FTC) requires that advertising claims for homeopathic drugs be backed by scientific evidence.<sup>[3]</sup><sup>[4]</sup>

 * *   **Warning Letters:** Since 2020, the FDA has issued numerous warning letters to companies for violations, including quality issues like sterility concerns.<sup>[2]</sup>

 * *   **Regulation as Drugs:** In the U.S., homeopathic products are regulated as drugs under the Federal Food, Drug, and Cosmetic Act.<sup>[4]</sup><sup>[3]</sup>

 * **2.<sup>[5]</sup>

 * *   **Implausibility:** Mainstream science does not accept the ideas behind homeopathy, such as "like-cures-like" and the increasing potency of highly diluted substances.<sup>[6]</sup>

 * *   **No Prevention of Disease:** There's no evidence that homeopathy can prevent diseases like malaria.<sup>[7]</sup>

 * *   **Homeopathy Research Institute (HRI):** This UK-based charity promotes research in homeopathy.<sup>[8]</sup>

 * *   **Advancements in Homeopathic Research:** This journal publishes research work and clinical studies in homeopathy.<sup>[5]</sup>

 * *   **Positive Replicated Experiments:** A review of basic research on highly dilute homeopathic medicines found a significant number of replicated experiments with positive results.<sup>[6]</sup>

 * *   **Nanoparticles:** There's growing evidence that nanoparticles play a role in how homeopathic medicines work.<sup>[6]</sup>

 * **4.<sup>[5]</sup>

 * *   **Reduced Use of Conventional Drugs:** Some research suggests that integrating homeopathy into medical practice may reduce the use of drugs like antibiotics.<sup>[6]</sup>

 * *   **Geographic Spread:** Homeopathy is geographically widespread and increasing in popularity, and has shown itself to be safe and effective for a range of conditions.<sup>[6]</sup>

 * *   **Royal Patronage:** King Charles III is a known supporter of homeopathy.<sup>[9]</sup>

 * *   **Homeopathy and Weight Loss:** Homeopathy is being explored as a gentle alternative to harsh medication for weight loss.<sup>[10]</sup>

 * *   **France:** France has abolished state funding for homeopathy.<sup>[11]</sup>

 * *   **New Homeopathy College:** A new private homeopathy college has been approved in Chittoor, India.<sup>[10]</sup>

 * *   **World Homeopathy Day:** World Homeopathy Day was observed on April 10, 2025.<sup>[10]</sup>

Citations:

1. [avalere.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAJDHK4-h1LHlW2iSwKSqtQyaZZdBpYh8Doya1q1ATEsIRDzCjpyBQ-NRjd5R8nSxqXqp2qxl4RNQ0aGAc0nfCdnUg-8IMw1My--q4ZOTDg6bxYeQ1kzNT8iNHgYgjK_fp9USVPYty-MsPPjqHwocvo_QiZopzEYHwQiKjutMdiO9cOJAZWNwbgrG_mBPUYT2fgrEA==)
2. [fda.gov](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAJUU98nzMk0Jex0iCkAxo5bY24mzF2vIbCBMuLiznjKDMtde4JC9p1EnS-OmjC4n0kFoFTdNqY43LPqCxIQpJqbC28cF53b-4GfODLVGuAZqyWtXGg2VPZECS0uzrm2jXcstdcAvrqk3JyWj_bVfHGaoYQ_x4zo_NwMks1MaNMq3g==)
3. [mintz.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAI2dt5-wxP51byAmNsRDlOJwSr-FC8E156wEK_Ge7eC2dlwnFWXu21tI6ktBk7dCuDHiN0b_zPa43V8jqgDeiRVo5tMGYYMnfo0bbAXpL9TStqlxkwk798J5UmodgcwLkKEu3Y8bf_kxLSGRyKUVTvFydEh0Xc6JiRSqL3MaL56AsPx-vXfjAyb2fLfPbTfGrmBLkBCmqhF1zmpt-jNI3zrzWkjTsx_H2EaC37fH1SWd2I=)
4. [nih.gov](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAL9XwzNDInFnSRaiWr8h_fMoS8fZYGPn5lJ_3xieYAAAJJtXNaYXXez9PxbiVBLr_vQXKT2rLyFv0mKV6N5GRYMIJ0fJXhO8Ql046MCmlBnTHy4bQ42bcTTKcK42Il_oI1hLQzgf7c=)
5. [acspublisher.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAIABooVMhGB12SjbKApGQYqRnLTaRxXGGMJJLTKmKxnXzGTWpsP7omag1Cfvo5wraW-Dmo1GTuzhSeoTSeR1rt1sxhAXGGqMvqvhhDnPLg26XBk445RBi5NYI9gWW8pQX5OHJWAxr812gOs)
6. [facultyofhomeopathy.org](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAINiBoDfsJmsmtpcY2i8-zU142hWbxFCS4GNnYxEZ64ud9xtC0-XtkPvtBtT_YAqiRU89EQNCtpO6nZ4CCL1yt__0G3KDc4vsXz7sG3ksGvqIQBj7WVZ8VQORiDm33krv6uaiI3FWGIzYlHeWrX8lI9)
7. [www.nhs.uk](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAKixMM9byqcd1X350sI1T7HNIxNG5vjdjjshmhEbbaGbBcvF4tfsnVhkFd6klJQZlb1kSE7UuTt7qa0bcEGt3I2xtOvS9n9xYNp_U3AJhP3vo3zuo5FxaxyzMZ3STS3yTZ69TE4)
8. [hri-research.org](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAJILUyQTCognEGOZQFpdKJPkH5OiT6sBrwL7mF7CL3r0lhDgLszWhWxrajaRFnAMcghz_NnTKHP1bMgY8WAyKPD0nwkKS2GekdQUVUzcjAG04KmC7z_LrU-)
9. [facultyofhomeopathy.org](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqALqcIj1oYKBrjIR6EUSATsij0QOzLWgQCBFMbw0G3SGdWsNYz3YCAViERsy2W9wjVZ2zy-3yUZAHYp2FXomEyNg0rlblamO-Lmm0lVJlCQl0Rmp8kMehSPi1RNlqWruZY3MDvxqd2KEIEholSAwfBjKBqVUh9LqOAr3GmQ=)
10. [indiatimes.com](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqALZVQP9a7ROLhSGuSaYtcLc3dF7Stxatr9axHmWYeCrj9w1TctTwLVsxFG1CixtU00lRKtMVmIAGWlkaXxzMPtvJ9PCLmnWr-sJ96pLCaK1smrYKqwDInPp8phY6OFjZZpRoqrfixy8R_lCYHYL3b59ERjN5Q==)
11. [independent.co.uk](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AWQVqAK7l7kHidSCg5glYqpW02EsJSOEGMmKcGkXtZy51UQPIIjMQMPCzujdFhq8mRQhfXQznzBkMhne8TFDy72-7ndP8XuQ74PbLs9G23tZOnuoNx94bRRfroOuyDqB1TWecRt_VuPjMKrWKkk=)


# Search with tools
We'll use enable the Google Search grounding tool and the code generation tool across two steps. In the first step, the model will use Google Search to find the requested information and then in the follow-up question, it generates code to plot the results.

This usage includes textual, visual and code parts, so first define a function to help visualise these.

In [39]:
from IPython.display import display, Image, Markdown

def show_response(response):
    for p in response.candidates[0].content.parts:
        if p.text:
            display(Markdown(p.text))
        elif p.inline_data:
            display(Image(p.inline_data.data))
        else:
            print(p.to_json_dict())
    
        display(Markdown('----'))

Now we start a chat asking for some information. Here you provide the Google Search tool so that the model can look up data from Google's Search index.

In [40]:
config_with_search = types.GenerateContentConfig(
    tools=[types.Tool(google_search=types.GoogleSearch())],
    temperature=1,
)

chat = client.chats.create(model='gemini-2.0-flash')

response = chat.send_message(
    message="Duration for recovery by homeopathy",
 #   config=config_with_search,
)

show_response(response)

The duration of recovery with homeopathy is a complex and debated topic. Here's a breakdown of the factors involved and different perspectives:

**Factors Influencing Recovery Time in Homeopathy:**

*   **Nature of the Illness:**
    *   **Acute conditions (e.g., colds, flu, minor injuries):**  Recovery can be relatively quick, sometimes within hours or days, if the correct remedy is found.
    *   **Chronic conditions (e.g., eczema, arthritis, anxiety, IBS):**  Recovery is generally slower and more gradual, taking weeks, months, or even longer.  Long-standing and deep-seated chronic conditions may require a longer treatment period.
    *   **Severity of the Illness:** More severe or advanced stages of an illness will naturally take longer to heal.
*   **Individual Factors:**
    *   **Vitality:**  A person with strong vitality (overall health, energy, and resilience) will generally respond more quickly to treatment.
    *   **Age:**  Children and younger adults often respond faster than older individuals.
    *   **Lifestyle:**  Factors like diet, exercise, stress levels, and exposure to toxins can significantly impact healing.
    *   **Underlying Conditions:**  The presence of other health issues can complicate and slow down the recovery process.
*   **Homeopathic Treatment Factors:**
    *   **Correct Remedy:** Finding the *similimum* (the most similar remedy) is crucial. If the remedy isn't a good match, improvement will be minimal or non-existent.  Finding the right remedy may involve multiple consultations and adjustments.
    *   **Potency and Dosage:**  The potency (strength) and frequency of the remedy need to be carefully tailored to the individual and the condition.  These may be adjusted over time.
    *   **Homeopath's Skill and Experience:** The homeopath's ability to accurately assess the case, select the appropriate remedy, and manage the treatment process is a major factor.
*   **Adherence to Treatment:** Following the homeopath's instructions regarding dosage, lifestyle adjustments, and follow-up appointments is essential.
*   **Interferences:** Certain substances (e.g., coffee, strong aromatic substances) or medical treatments (e.g., certain medications) may interfere with the action of the homeopathic remedy.

**General Expectations:**

*   **Acute Conditions:** Noticeable improvement within 24-48 hours is often expected with a well-chosen remedy.  If no improvement is seen within this timeframe, the remedy may need to be re-evaluated.
*   **Chronic Conditions:**
    *   **Initial Phase:**  In the early stages of treatment, there may be subtle shifts in symptoms, energy levels, or overall well-being.  Sometimes, there can be an initial aggravation (a temporary worsening of symptoms) before improvement occurs.
    *   **Gradual Improvement:**  Over time, symptoms should gradually lessen in frequency and intensity.  There may be ups and downs during the healing process.
    *   **Maintenance:**  Once significant improvement is achieved, maintenance doses of the remedy may be needed to sustain the benefits.
    *   **Timeframe:** It's difficult to give a precise timeframe for chronic conditions. Some individuals may experience significant improvement within a few months, while others may require a year or more of consistent treatment.

**What to Discuss with Your Homeopath:**

*   **Realistic Expectations:**  Have an open discussion with your homeopath about what you can realistically expect in terms of recovery time, given your specific condition and circumstances.
*   **Monitoring Progress:**  Establish a system for monitoring your progress and reporting back to your homeopath on a regular basis.
*   **Treatment Plan:**  Get a clear understanding of the treatment plan, including the remedy, potency, dosage, and any lifestyle recommendations.
*   **When to Seek Conventional Care:**  Discuss when it would be appropriate to seek conventional medical care if your symptoms worsen or if you're not seeing improvement with homeopathy.

**Important Considerations:**

*   **Homeopathy is Individualized:**  Recovery times vary greatly from person to person.
*   **Commitment:** Homeopathic treatment often requires a commitment to lifestyle changes and a willingness to work closely with your homeopath.
*   **Research:** If you are considering homeopathic treatment, it is advisable to research the practitioner's credentials and experience.
*   **Integration with Conventional Medicine:** Homeopathy can be used alongside conventional medicine, but it is crucial to inform both your homeopath and your conventional doctor about all treatments you are receiving.

**In Conclusion:**

Recovery time in homeopathy is variable and depends on many factors.  While acute conditions may resolve relatively quickly, chronic conditions typically require a more gradual and longer-term approach.  Open communication with your homeopath, realistic expectations, and adherence to the treatment plan are key to a successful outcome.


----

Thus we can get latest updated information about Homeopathic Medicines